In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
from ollama import Client
from IPython.display import Audio
from kokoro import KPipeline
import soundfile as sf
import re
from qdrant_client import QdrantClient
from io import BytesIO
from pydub import AudioSegment
import sys
import os
import language_tool_python
import re

sys.path.append(os.path.abspath('../Backend'))

from generate_meditation import generate_text, parsear_meditacion, ajustar_silencios_y_reconstruir
# Configuration


In [2]:
tool = language_tool_python.LanguageTool('es')

In [ ]:
texto = generate_text(15, "principiante")

In [5]:
segmentos = parsear_meditacion(texto)
pipeline = KPipeline(lang_code='e')

/home/juanjo/Projects/Contemplative-ai/contemplative-ai/venv/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
/home/juanjo/Projects/Contemplative-ai/contemplative-ai/venv/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [6]:
def generar_audio_segmentos(segmentos, pipeline):
    audio_texto = []
    dur_texto_muestras = 0
    dur_silencios_segundos = 0
    for tipo, contenido in segmentos:
        if tipo == 'texto':
            #logger.info(f"Generando audio para texto: {contenido}")
            generator = pipeline(contenido, voice="ef_dora", speed=0.8)
            audio = np.concatenate([audio for _, _, audio in generator])
            audio_texto.append(audio)
            dur_texto_muestras += len(audio)
        elif tipo == 'silencio':
            dur_silencios_segundos += contenido
    return audio_texto, dur_texto_muestras, dur_silencios_segundos

In [7]:
audio_texto, dur_texto_muestras, dur_silencios_segundos = generar_audio_segmentos(segmentos, pipeline)

In [8]:
audio_final = ajustar_silencios_y_reconstruir(segmentos, audio_texto, dur_texto_muestras, dur_silencios_segundos, 5 * 60)

In [9]:
output_path = f"../data/audio/meditacion_kokoro_test.wav"
sf.write(output_path, audio_final, samplerate=24000)